## Import các thư viện cần thiết

In [9]:
import pandas as pd
import numpy as np
import re
import os

## Thu thập dữ liệu

In [10]:
df = pd.read_csv('../data/raw/batdongsan_raw.csv')

print(f"Số lượng dòng: {len(df)}")
print(f"\nCác cột: {df.columns.tolist()}")
print(f"\nThông tin dữ liệu:")
print(df.info())
print(f"\nThông tin 5 dòng đầu tiên:")
print(df.head().to_string())

# Thay thế 'N/A' bằng NaN để xử lý thống nhất
df = df.replace('N/A', np.nan)

Số lượng dòng: 10107

Các cột: ['tieu_de', 'gia', 'dia_chi', 'dien_tich_dat', 'phong_ngu', 'phong_tam', 'so_tang', 'phap_ly', 'ngay_dang']

Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10107 entries, 0 to 10106
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   tieu_de        10106 non-null  object
 1   gia            10107 non-null  object
 2   dia_chi        10107 non-null  object
 3   dien_tich_dat  10107 non-null  object
 4   phong_ngu      8194 non-null   object
 5   phong_tam      7728 non-null   object
 6   so_tang        9506 non-null   object
 7   phap_ly        9051 non-null   object
 8   ngay_dang      10107 non-null  object
dtypes: object(9)
memory usage: 710.8+ KB
None

Thông tin 5 dòng đầu tiên:
                                                                                               tieu_de      gia                                                   dia_chi dien_tich_dat p

## Cách thức tiền xử lý:

*   **`tieu_de`**: Giữ nguyên kiểu dữ liệu `object` (string), chỉ thực hiện làm sạch bằng cách loại bỏ các ký tự xuống dòng và khoảng trắng thừa.

*   **`gia`**: Chuyển từ kiểu dữ liệu `object` (string) sang `float64` (số thực). Ví dụ chuỗi "2,5 tỷ" sẽ chuyển thành 2.5.

*   **`dia_chi`**: Giữ nguyên kiểu dữ liệu `object` (string).

*   **`dien_tich_dat`**: Chuyển từ kiểu dữ liệu `object` (string) sang `float64` (số thực). Ví dụ chuỗi "125,5 m²" sẽ chuyển thành 125.5.

*   **`phong_ngu`**: Chuyển từ kiểu dữ liệu `object` (string) sang `Int64` (số nguyên). Ví dụ chuỗi "3 phòng" chuyển thành 3.

*   **`phong_tam`**: Chuyển từ kiểu dữ liệu `object` (string) sang `Int64` (số nguyên). Ví dụ chuỗi "3 phòng" chuyển thành 3.

*   **`so_tang`**: Chuyển từ kiểu dữ liệu `object` (string) sang `Int64` (số nguyên). Ví dụ chuỗi "2 tầng" chuyển thành 2.

*   **`phap_ly`**: Giữ nguyên kiểu dữ liệu `object` (string) nhưng chuỗi chuẩn hóa thành một trong ba loại: "Sổ hồng", "Sổ đỏ", hoặc "Giấy tờ không xác định".

*   **`ngay_dang`**: Chuyển từ kiểu dữ liệu `object` (string) sang `datetime64[ns]`. Ví dụ chuỗi "dd/mm/yyyy" thành kiểu dữ liệu datetime (yyyy-mm-dd).

## Tiền xử lý

In [11]:
def preprocess_title(title_str):
    if pd.isna(title_str):
        return np.nan # Giữ nguyên NaN nếu giá trị bị thiếu
    
    # Chuyển đổi sang chuỗi và thay thế '\n' bằng khoảng trắng
    # Sau đó loại bỏ các khoảng trắng thừa ở đầu và cuối chuỗi
    return str(title_str).replace('\n', ' ').strip()

# Chuyển đổi giá từ chuỗi sang số (đơn vị: tỷ VNĐ)
def preprocess_price(price_str):
    if pd.isna(price_str):
        return np.nan
    
    # Kiểm tra nếu giá trị là "Thỏa thuận" thì trả về NaN
    price_str_stripped = str(price_str).strip()
    if price_str_stripped.lower() == 'thỏa thuận':
        return np.nan
    
    price_str = str(price_str).lower().replace(' ', '').replace(',', '.')
    
    try:
        if 'tỷ' in price_str:
            # Lấy số và giữ nguyên đơn vị tỷ
            num = float(re.findall(r'\d+\.?\d*', price_str)[0])
        elif 'triệu' in price_str:
            # Chuyển triệu sang tỷ (chia cho 1000)
            num = float(re.findall(r'\d+\.?\d*', price_str)[0]) / 1000
        else:
            return np.nan
        
        return num if num > 0 else np.nan
    except:
        return np.nan

# Chuyển đổi diện tích từ chuỗi sang số thực (m²)    
def preprocess_area(area_str):
    if pd.isna(area_str):
        return np.nan
    try:
        # Trích xuất số từ chuỗi (ví dụ: "32 m²" -> 32.0)
        standardized_str = str(area_str).replace(',', '.')
        # Trích xuất số từ chuỗi đã được chuẩn hóa
        num = float(re.findall(r'\d+\.?\d*', standardized_str)[0])
        return num if num > 0 else np.nan
    except:
        return np.nan

# Chuyển đổi số phòng (phòng ngủ, phòng tắm) từ chuỗi sang số nguyên
def preprocess_rooms(room_str):
    if pd.isna(room_str):
        return np.nan
    
    try:
        # Trích xuất số từ chuỗi (ví dụ: "3 phòng" -> 3)
        num = int(re.findall(r'\d+', str(room_str))[0])
        return num if num >= 0 else np.nan
    except:
        return np.nan

# Chuyển đổi số tầng từ chuỗi sang số nguyên
def preprocess_floors(floor_str):
    if pd.isna(floor_str):
        return np.nan
    
    try:
        # Trích xuất số nguyên từ chuỗi (ví dụ: "3 tầng" -> 3)
        num = int(re.findall(r'\d+', str(floor_str))[0])
        return num if num > 0 else np.nan
    except:
        return np.nan
    
# Chuyển đổi thông tin pháp lý thành 3 dạng: Sổ hồng, Sổ đỏ, Giấy tờ không xác định
def preprocess_legal_status(phap_ly_str):
    if pd.isna(phap_ly_str):
        return np.nan
    
    phap_ly_lower = str(phap_ly_str).lower().strip()

    # Trường hợp "Sổ đỏ/ Sổ hồng" 
    if re.search(r'sổ\s*đỏ\s*[/\\&]\s*sổ\s*hồng', phap_ly_lower):
        return "Sổ hồng"
    # Trường hợp "Sổ hồng" (bao gồm "sổ hồng riêng", "có sổ hồng",...)
    elif re.search(r'sổ\s*hồng', phap_ly_lower):
        return "Sổ hồng"
    # Trường hợp "Sổ đỏ"
    elif re.search(r'sổ\s*đỏ', phap_ly_lower):
        return "Sổ đỏ"
    # Trường hợp "Có sổ." hoặc bất kỳ trường hợp nào khác không khớp
    else:
        return "Giấy tờ không xác định"

# Chuyển đổi ngày đăng từ chuỗi dd/mm/yyyy sang datetime (yyyy-mm-dd)     
def preprocess_date(date_str):
    """Chuyển đổi ngày đăng sang định dạng dd/mm/yyyy"""
    if pd.isna(date_str):
        return np.nan
    try:
        # Chuyển sang datetime 
        dt = pd.to_datetime(date_str, format='%d/%m/%Y')
        return dt
    except:
        return np.nan

## Thực hiện tiền xử lý

In [12]:
print("Bắt đầu tiền xử lý dữ liệu\n")

# Tạo bản sao
df_processed = df.copy()

# Giữ nguyên tiêu đề
df_processed['tieu_de'] = df_processed['tieu_de'].apply(preprocess_title)

# Xử lý giá
df_processed['gia'] = df_processed['gia'].apply(preprocess_price)

# Xử lý diện tích
df_processed['dien_tich_dat'] = df_processed['dien_tich_dat'].apply(preprocess_area)

# Xử lý phòng ngủ
df_processed['phong_ngu'] = df_processed['phong_ngu'].apply(preprocess_rooms)
df_processed['phong_ngu'] = pd.to_numeric(df_processed['phong_ngu'], errors='coerce').astype('Int64')

# Xử lý phòng tắm
df_processed['phong_tam'] = df_processed['phong_tam'].apply(preprocess_rooms)
df_processed['phong_tam'] = pd.to_numeric(df_processed['phong_tam'], errors='coerce').astype('Int64')
# Xử lý số tầng
df_processed['so_tang'] = df_processed['so_tang'].apply(preprocess_floors)
df_processed['so_tang'] = pd.to_numeric(df_processed['so_tang'], errors='coerce').astype('Int64')

# Xử lý pháp lý
df_processed['phap_ly'] = df_processed['phap_ly'].apply(preprocess_legal_status)

# Xử lý ngày đăng
df_processed['ngay_dang'] = df_processed['ngay_dang'].apply(preprocess_date)

before_count = len(df_processed)

required_cols = ['tieu_de', 'gia', 'dia_chi', 'dien_tich_dat', 'phong_ngu', 
                 'phong_tam', 'so_tang', 'phap_ly', 'ngay_dang']
df_processed = df_processed.dropna(subset=required_cols)

after_count = len(df_processed)
print(f"Đã loại bỏ {before_count - after_count} dòng thiếu dữ liệu")
print(f"Còn lại: {after_count} dòng")

print("\nLoại bỏ trùng lặp:")
before_count = len(df_processed)

# Loại bỏ trùng lặp dựa trên tiêu đề và địa chỉ
df_processed = df_processed.drop_duplicates(subset=['tieu_de', 'dia_chi'], keep='first')

after_count = len(df_processed)
print(f"Đã loại bỏ {before_count - after_count} dòng trùng lặp")
print(f"Còn lại: {after_count} dòng\n")

# Kiểm tra kiểu dữ liệu
print("Kiểu dữ liệu của các cột sau khi tiền xử lý: ")
print(df_processed.dtypes)

print("\nThông tin 5 dòng đầu tiên sau khi tiền xử lý: \n")

print(df_processed.head().to_string())

# Đường dẫn file đầu ra
output_processed_path = '../data/interim/batdongsan_interim.csv'

# Tạo thư mục nếu nó chưa tồn tại
os.makedirs(os.path.dirname(output_processed_path), exist_ok=True)

# Lưu DataFrame đã xử lý vào file CSV
df_processed.to_csv(output_processed_path, index=False, encoding='utf-8-sig')

print(f"\nDữ liệu đã được tiền xử lý và lưu vào file: '{output_processed_path}'")

Bắt đầu tiền xử lý dữ liệu

Đã loại bỏ 3372 dòng thiếu dữ liệu
Còn lại: 6735 dòng

Loại bỏ trùng lặp:
Đã loại bỏ 48 dòng trùng lặp
Còn lại: 6687 dòng

Kiểu dữ liệu của các cột sau khi tiền xử lý: 
tieu_de                  object
gia                     float64
dia_chi                  object
dien_tich_dat           float64
phong_ngu                 Int64
phong_tam                 Int64
so_tang                   Int64
phap_ly                  object
ngay_dang        datetime64[ns]
dtype: object

Thông tin 5 dòng đầu tiên sau khi tiền xử lý: 

                                                                                              tieu_de    gia                                                       dia_chi  dien_tich_dat  phong_ngu  phong_tam  so_tang  phap_ly  ngay_dang
1    Cc bán nhà hẻm 97/ Phan Đăng Lưu Quận Phú Nhuận dt 6 x 8m dtcn 44,5m2 k/c trệt 2 lầu st giá 9 tỷ   9.00      Phố Phan Đăng Lưu, Phường 7, Quận Phú Nhuận, Hồ Chí Minh           44.5          4          5        